In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import time
import logging
import sys
import json
from datetime import datetime
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 从CoFrNet_DL.py中导入必要的类
class SingleLadder(nn.Module):
    def __init__(self, depth, epsilon=0.1):
        super(SingleLadder, self).__init__()
        self.depth = depth
        self.epsilon = epsilon
        self.weights = nn.Parameter(torch.randn(depth + 1) * 0.1)

    def safe_reciprocal(self, x):
        sign = torch.sign(x)
        abs_x = torch.abs(x)
        safe_abs = torch.clamp(abs_x, min=self.epsilon)
        return sign / safe_abs

    def forward(self, x_j):
        if self.depth == 0:
            return self.weights[0] * x_j
        
        result = self.weights[self.depth] * x_j
        
        for k in range(self.depth - 1, 0, -1):
            denominator = self.weights[k] * x_j + self.safe_reciprocal(result)
            result = denominator
        
        final_result = self.weights[0] * x_j + self.safe_reciprocal(result)
        return final_result

class FullLadder(nn.Module):
    def __init__(self, input_dim, depth, epsilon=0.1):
        super(FullLadder, self).__init__()
        self.input_dim = input_dim
        self.depth = depth
        self.epsilon = epsilon
        self.weights = nn.Parameter(torch.randn(depth + 1, input_dim) * 0.1)
        
    def safe_reciprocal(self, x):
        sign = torch.sign(x)
        abs_x = torch.abs(x)
        safe_abs = torch.clamp(abs_x, min=self.epsilon)
        return sign / safe_abs

    def forward(self, x):
        if self.depth == 0:
            return F.linear(x, self.weights[0])

        result = F.linear(x, self.weights[self.depth])
        
        for k in range(self.depth - 1, 0, -1):
            a_k = F.linear(x, self.weights[k])
            denominator = a_k + self.safe_reciprocal(result)
            result = denominator
            
        a_0 = F.linear(x, self.weights[0])
        final_result = a_0 + self.safe_reciprocal(result)
        return final_result

class CoFrNetDLRegression(nn.Module):
    """回归任务专用的CoFrNet-DL"""
    def __init__(self, input_dim, num_full_ladders=25, max_diag_ladder_depth=12, max_full_ladder_depth=12, epsilon=0.1):
        super(CoFrNetDLRegression, self).__init__()
        self.input_dim = input_dim
        self.num_full_ladders = num_full_ladders
        self.max_diag_ladder_depth = max_diag_ladder_depth
        self.max_full_ladder_depth = max_full_ladder_depth
        
        # 1. 对角梯子部分
        self.diag_ladders = nn.ModuleList()
        self.diag_ladder_depths = []
        
        for i in range(input_dim):
            depth = min(max_diag_ladder_depth, max(2, 5 + (i * 2) % 10))
            self.diag_ladder_depths.append(depth)
            self.diag_ladders.append(SingleLadder(depth, epsilon))
            
        # 2. 全连接梯子部分
        self.full_ladders = nn.ModuleList()
        self.full_ladder_depths = []
        
        for i in range(num_full_ladders):
            if num_full_ladders > 1:
                depth = 2 + int((i / (num_full_ladders - 1)) * (max_full_ladder_depth - 2))
            else:
                depth = 2
            depth = min(max_full_ladder_depth, max(2, depth))
            self.full_ladder_depths.append(depth)
            self.full_ladders.append(FullLadder(input_dim, depth, epsilon))
            
        # 3. 最终线性组合层 - 回归任务输出维度为1
        combined_input_dim = input_dim + num_full_ladders
        self.output_layer = nn.Linear(combined_input_dim, 1)
        
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
        
    def forward(self, x):
        batch_size = x.shape[0]
        
        # 1. 计算对角梯子输出
        diag_outputs = []
        for j in range(self.input_dim):
            x_j = x[:, j]  
            ladder_output = self.diag_ladders[j](x_j)
            diag_outputs.append(ladder_output.unsqueeze(1))
        
        diag_features = torch.cat(diag_outputs, dim=1)  
        
        # 2. 计算全连接梯子输出
        full_outputs = []
        for ladder in self.full_ladders:
            ladder_output = ladder(x)  
            full_outputs.append(ladder_output.unsqueeze(1))
        
        full_features = torch.cat(full_outputs, dim=1)  
        
        # 3. 拼接所有特征
        combined_features = torch.cat([diag_features, full_features], dim=1)
        
        # 4. 最终线性组合
        output = self.output_layer(combined_features)
        return output.squeeze()  # 回归任务输出一维

# --- 1. 配置与环境设置 ---
def setup_logging(log_file='training_regression_cofrnet_dl.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger(); root_logger.handlers = []; root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w'); file_handler.setFormatter(log_formatter); root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout); console_handler.setFormatter(log_formatter); root_logger.addHandler(console_handler)

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. 数据准备 ---
def get_regression_data_loaders(batch_size=128, test_size=0.3):
    """准备在线新闻流行度回归数据集"""
    logging.info("正在准备在线新闻流行度回归数据集...")
    
    # 加载在线新闻流行度数据集
    df = pd.read_csv('/kaggle/input/online-news-popularity/OnlineNewsPopularity.csv')
    
    # 删除无用和共线性特征
    cols_to_drop = ['url', ' timedelta', 
                   ' abs_title_sentiment_polarity', ' LDA_04', ' is_weekend',
                   ' rate_positive_words', ' abs_title_subjectivity', 
                   ' rate_negative_words']
    df = df.drop(cols_to_drop, axis=1)
    
    # 特征和目标变量
    X = df.drop([' shares'], axis=1)  # 特征矩阵
    y = df[' shares']  # 目标变量（文章分享次数）
    
    # 数据标准化
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # 对目标变量取对数并标准化（因为分享次数通常呈长尾分布）
    y = np.log1p(y)  # 使用log(1+y)转换
    y = (y - y.mean()) / y.std()  # 标准化目标变量
    
    # 划分训练测试集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    # 转换为PyTorch张量
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train.values)  
    X_test = torch.FloatTensor(X_test)
    y_test = torch.FloatTensor(y_test.values)   
   
    # 创建DataLoader
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
    
    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    logging.info("在线新闻流行度回归数据集准备完毕。")
    logging.info(f"训练样本数: {len(train_dataset)}, 测试样本数: {len(test_dataset)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")
    
    return trainloader, testloader, X_train.shape[1]

# --- 3. 训练函数 ---
def train_regression(model, trainloader, testloader, hparams, log_filepath, model_save_path):
    criterion = nn.MSELoss()  # 回归任务使用MSE损失
    optimizer = optim.Adam(model.parameters(), 
                         lr=hparams['learning_rate'], 
                         weight_decay=hparams['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=hparams['patience']//3, factor=0.5)
    
    best_test_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "hyperparameters": hparams,
        "training_losses": [],
        "test_losses": [],
        "learning_rates": []
    }

    logging.info("="*30)
    logging.info("  启动 CoFrNet-DL (回归版) 训练流程")
    logging.info("="*30)
    
    num_epochs = hparams['num_epochs']
    
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.float().to(device)
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        epoch_avg_loss = running_loss / len(trainloader)
        log_data["training_losses"].append(epoch_avg_loss)
        log_data["learning_rates"].append(optimizer.param_groups[0]['lr'])
        
        # 验证阶段
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = model(inputs)
                test_loss += criterion(outputs, targets).item()
        
        avg_test_loss = test_loss / len(testloader)
        log_data["test_losses"].append(avg_test_loss)
        
        scheduler.step(avg_test_loss)
        
        logging.info(f"Epoch {epoch+1}/{num_epochs} | 训练MSE损失: {epoch_avg_loss:.4f} | 测试MSE损失: {avg_test_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        # 早停检查
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            patience_counter = 0
            best_model_state = model.state_dict()
            logging.info(f"*** 新的最佳MSE损失: {best_test_loss:.4f} ***")
        else:
            patience_counter += 1
            if patience_counter >= hparams['patience']:
                logging.info(f"早停: 连续 {hparams['patience']} 轮无改进")
                break
        
        # 保存日志
        with open(log_filepath, 'w') as f:
            json.dump(log_data, f, indent=4)
        
        if best_test_loss < hparams['loss_target']:
            logging.info(f"目标损失 {hparams['loss_target']} 已达到。")
            break

    if best_model_state:
        logging.info(f"训练流程结束。保存最佳模型到 {model_save_path}")
        torch.save({
            'model_state_dict': best_model_state,
            'best_loss': best_test_loss,
            'hyperparameters': hparams
        }, model_save_path)
    else:
        logging.error("没有训练出任何有效模型，不进行保存。")

    return best_test_loss, best_model_state

# --- 4. 主程序 ---
if __name__ == '__main__':
    # 文件与超参数定义
    LOG_FILE = 'training_regression_cofrnet_dl.log'
    JSON_LOG_FILE = 'training_regression_cofrnet_dl_log.json'
    MODEL_SAVE_PATH = 'regression_cofrnet_dl.pth'
    
    HYPERPARAMETERS = {
        "num_epochs": 50,
        "loss_target": 0.1,  # MSE目标值
        "patience": 15,
        "learning_rate": 1e-3,
        "weight_decay": 1e-5,
        "num_full_ladders": 25,
        "max_diag_ladder_depth": 12,
        "max_full_ladder_depth": 12
    }

    setup_logging(LOG_FILE)
    logging.info(f"回归实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    trainloader, testloader, input_dim = get_regression_data_loaders()
    
    # 创建CoFrNet-DL回归模型
    model = CoFrNetDLRegression(
        input_dim=input_dim,
        num_full_ladders=HYPERPARAMETERS['num_full_ladders'],
        max_diag_ladder_depth=HYPERPARAMETERS['max_diag_ladder_depth'],
        max_full_ladder_depth=HYPERPARAMETERS['max_full_ladder_depth']
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    logging.info(f"模型总参数数量: {total_params}")

    start_time = time.time()
    best_loss, best_state_dict = train_regression(
        model, trainloader, testloader,
        hparams=HYPERPARAMETERS,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")
    
    # 加载并验证保存的最佳模型
    if best_state_dict:
        logging.info("\n" + "="*30)
        logging.info("  开始验证已保存的最佳回归模型")
        logging.info("="*30)
        
        verification_model = CoFrNetDLRegression(
            input_dim=input_dim,
            num_full_ladders=HYPERPARAMETERS['num_full_ladders'],
            max_diag_ladder_depth=HYPERPARAMETERS['max_diag_ladder_depth'],
            max_full_ladder_depth=HYPERPARAMETERS['max_full_ladder_depth']
        ).to(device)
        
        checkpoint = torch.load(MODEL_SAVE_PATH)
        verification_model.load_state_dict(checkpoint['model_state_dict'])
        verification_model.eval()

        logging.info(f"成功加载模型。保存的MSE损失: {checkpoint['best_loss']:.4f}")

        test_loss = 0.0
        criterion = nn.MSELoss()
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = verification_model(inputs)
                test_loss += criterion(outputs, targets).item()
        
        avg_test_loss = test_loss / len(testloader)
        logging.info(f"加载后的模型在测试集上的MSE损失: {avg_test_loss:.4f}")
        
        assert abs(avg_test_loss - checkpoint['best_loss']) < 1e-3, "验证失败！"
        logging.info("验证成功！回归模型可以被正确保存和复用。")
    else:
        logging.info("未能成功训练和保存回归模型。")

2025-08-14 11:04:37,884 [INFO ]  回归实验超参数: {
    "num_epochs": 50,
    "loss_target": 0.1,
    "patience": 15,
    "learning_rate": 0.001,
    "weight_decay": 1e-05,
    "num_full_ladders": 25,
    "max_diag_ladder_depth": 12,
    "max_full_ladder_depth": 12
}
2025-08-14 11:04:37,885 [INFO ]  正在准备在线新闻流行度回归数据集...
2025-08-14 11:04:38,932 [INFO ]  在线新闻流行度回归数据集准备完毕。
2025-08-14 11:04:38,933 [INFO ]  训练样本数: 27750, 测试样本数: 11894
2025-08-14 11:04:38,934 [INFO ]  特征维度: 52, 目标维度: 1
2025-08-14 11:04:38,972 [INFO ]  模型总参数数量: 10410
2025-08-14 11:04:42,474 [INFO ]  ==============================
2025-08-14 11:04:42,475 [INFO ]    启动 CoFrNet-DL (回归版) 训练流程
2025-08-14 11:04:42,476 [INFO ]  ==============================
2025-08-14 11:05:10,546 [INFO ]  Epoch 1/50 | 训练MSE损失: 31.3381 | 测试MSE损失: 7.8751 | LR: 1.00e-03
2025-08-14 11:05:10,548 [INFO ]  *** 新的最佳MSE损失: 7.8751 ***
2025-08-14 11:05:37,944 [INFO ]  Epoch 2/50 | 训练MSE损失: 8.7791 | 测试MSE损失: 3.2246 | LR: 1.00e-03
2025-08-14 11:05:37,946 [INFO ]  *** 新的

AssertionError: 验证失败！